# Initialisation

In [2]:
import os
import json
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from google.colab import drive
# 1. Mount Drive
drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/asl-signs'
# 2. Load Data


Mounted at /content/drive


In [2]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)

In [3]:
print("Loading data...")
X = np.load(f'{BASE_DIR}/X.npy')
y = np.load(f'{BASE_DIR}/y.npy')
with open(f'{BASE_DIR}/index_to_sign.json', 'r') as f:
    index_to_sign = json.load(f)
NUM_FRAMES = 32
N_LANDMARKS = 75
NUM_CLASSES = len(index_to_sign)
BATCH_SIZE = 64


Loading data...


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/asl-signs/X.npy'

# Training 2

In [1]:
import numpy as np, tensorflow as tf, json, os, gc
from sklearn.model_selection import train_test_split
from google.colab import drive

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)

drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/asl-signs'
print("Ready")

Mounted at /content/drive
Ready


In [2]:
# Load directly as float16 — never create a float32 copy
X = np.load(f'{BASE_DIR}/X.npy', mmap_mode='r')
y = np.load(f'{BASE_DIR}/y.npy')

# Split indices only — don't copy data
from sklearn.model_selection import train_test_split
idx = np.arange(len(X))
idx_train, idx_val = train_test_split(idx, test_size=0.15, random_state=42, stratify=y)

X_train = X[idx_train].astype(np.float16)
X_val   = X[idx_val].astype(np.float16)
y_train = y[idx_train].astype(np.int32)
y_val   = y[idx_val].astype(np.int32)

del X, idx, idx_train, idx_val
gc.collect()

print(f"X_train: {X_train.shape} {X_train.nbytes/1e9:.2f} GB")
print(f"X_val:   {X_val.shape}   {X_val.nbytes/1e9:.2f} GB")

X_train: (80305, 32, 75, 3) 1.16 GB
X_val:   (14172, 32, 75, 3)   0.20 GB


In [4]:
NUM_FRAMES, N_LANDMARKS, NUM_CLASSES, BATCH_SIZE = 32, 75, 250, 64

def augment(x, y):
    x = tf.cast(x, tf.float32)
    x = x + tf.random.normal(tf.shape(x), stddev=0.02)
    x = tf.cond(tf.random.uniform(()) > 0.5,
                lambda: x * tf.constant([[[-1., 1., 1.]]]),
                lambda: x)
    return x, y

train_ds = (tf.data.Dataset.from_tensor_slices((X_train, y_train))
            .shuffle(10000).batch(BATCH_SIZE)
            .map(augment, num_parallel_calls=tf.data.AUTOTUNE)
            .repeat().prefetch(tf.data.AUTOTUNE))

val_ds = (tf.data.Dataset.from_tensor_slices((X_val, y_val))
          .batch(BATCH_SIZE)
          .map(lambda x, y: (tf.cast(x, tf.float32), y),
               num_parallel_calls=tf.data.AUTOTUNE)
          .prefetch(tf.data.AUTOTUNE))

print("Datasets ready")

Datasets ready


In [5]:
def build_model(num_classes=NUM_CLASSES):
    inp = tf.keras.Input(shape=(NUM_FRAMES, N_LANDMARKS, 3))
    x   = tf.keras.layers.Reshape((NUM_FRAMES, N_LANDMARKS*3))(inp)
    x   = tf.keras.layers.Dense(256, activation='relu')(x)
    x   = tf.keras.layers.Dropout(0.3)(x)
    x   = tf.keras.layers.Bidirectional(
              tf.keras.layers.LSTM(256, return_sequences=True, dropout=0.3))(x)
    x   = tf.keras.layers.Bidirectional(
              tf.keras.layers.LSTM(128, dropout=0.3))(x)
    x   = tf.keras.layers.Dense(256, activation='relu',
              kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x   = tf.keras.layers.Dropout(0.4)(x)
    out = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
    return tf.keras.Model(inp, out)

model = build_model()
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 32, 75, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 32, 225)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32, 256)        │        57,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 32, 512)        │     1,050,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 256)            │       656,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 250)            │        64,250 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,894,906 (7.23 MB)

 Trainable params: 1,894,906 (7.23 MB)

 Non-trainable params: 0 (0.00 B)

In [6]:
for xb, yb in train_ds.take(1):
    with tf.GradientTape() as tape:
        preds = model(xb, training=True)
        loss  = tf.reduce_mean(
                    tf.keras.losses.sparse_categorical_crossentropy(yb, preds))
    grads = tape.gradient(loss, model.trainable_variables)
    norms = [float(tf.norm(g)) for g in grads if g is not None]
    print(f"Loss:           {float(loss):.4f}")
    print(f"Grad norms:     {[f'{n:.4f}' for n in norms]}")
    print(f"Any zero grads: {any(n == 0.0 for n in norms)}")

Loss:           5.5765
Grad norms:     ['0.4751', '0.0362', '0.5883', '0.1859', '0.0454', '0.6025', '0.1911', '0.0472', '0.4593', '0.1830', '0.0706', '0.4596', '0.1806', '0.0668', '0.4073', '0.1124', '0.4089', '0.1238']
Any zero grads: False


In [8]:
# Compare what two very different signs look like after normalization
tv_idx    = np.where(y_train == 0)[0][0]   # TV
why_idx   = np.where(y_train == 240)[0][0] # why

tv  = X_train[tv_idx].astype(np.float32)
why = X_train[why_idx].astype(np.float32)

print("TV  — mean: {:.4f}  std: {:.4f}  min: {:.4f}  max: {:.4f}".format(
    tv.mean(), tv.std(), tv.min(), tv.max()))
print("WHY — mean: {:.4f}  std: {:.4f}  min: {:.4f}  max: {:.4f}".format(
    why.mean(), why.std(), why.min(), why.max()))

# Check per-landmark variance across all TV samples
tv_indices = np.where(y_train == 0)[0]
tv_samples = X_train[tv_indices].astype(np.float32)
print(f"\nTV samples shape: {tv_samples.shape}")
print(f"Variance across TV samples (should be HIGH if sign is consistent):")
print(f"  per-landmark std mean: {tv_samples.std(axis=0).mean():.4f}")

# Compare against variance WITHIN a single sample
print(f"  within-sample std mean: {np.array([tv_samples[i].std() for i in range(len(tv_samples))]).mean():.4f}")

TV  — mean: 0.0458  std: 0.8359  min: -3.2793  max: 4.1797
WHY — mean: -0.0981  std: 0.8655  min: -3.7129  max: 3.0801

TV samples shape: (327, 32, 75, 3)
Variance across TV samples (should be HIGH if sign is consistent):
  per-landmark std mean: 0.2752
  within-sample std mean: 0.8492


In [12]:
# ── Learning rate warmup + cosine decay ──────────────────────────────────
import math

EPOCHS = 100
WARMUP_EPOCHS = 5
steps_per_epoch = len(X_train) // BATCH_SIZE
total_steps = EPOCHS * steps_per_epoch
warmup_steps = WARMUP_EPOCHS * steps_per_epoch

class WarmupCosineDecay(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, peak_lr, total_steps, warmup_steps):
        self.peak_lr     = peak_lr
        self.total_steps = total_steps
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warmup_lr = self.peak_lr * (step / self.warmup_steps)
        cosine_lr = self.peak_lr * 0.5 * (
            1 + tf.cos(math.pi * (step - self.warmup_steps)
                       / (self.total_steps - self.warmup_steps)))
        return tf.where(step < self.warmup_steps, warmup_lr, cosine_lr)

    def get_config(self):
        return {'peak_lr': self.peak_lr,
                'total_steps': self.total_steps,
                'warmup_steps': self.warmup_steps}

lr_schedule = WarmupCosineDecay(
    peak_lr=1e-3,
    total_steps=total_steps,
    warmup_steps=warmup_steps
)

# ── Fresh model — no LayerNorm, reduced dropout for faster warmup ─────────
def build_model(num_classes):
    inp = tf.keras.Input(shape=(32, 75, 3))
    x   = tf.keras.layers.Reshape((32, 75*3))(inp)
    x   = tf.keras.layers.Dense(256, activation='relu')(x)
    x   = tf.keras.layers.Dropout(0.2)(x)                        # reduced from 0.3
    x   = tf.keras.layers.Bidirectional(
              tf.keras.layers.LSTM(256, return_sequences=True))(x)  # removed LSTM dropout
    x   = tf.keras.layers.Bidirectional(
              tf.keras.layers.LSTM(128))(x)                         # removed LSTM dropout
    x   = tf.keras.layers.Dense(256, activation='relu',
              kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x   = tf.keras.layers.Dropout(0.3)(x)                        # reduced from 0.4
    out = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
    return tf.keras.Model(inp, out)

model = build_model(NUM_CLASSES)
model.compile(
    optimizer=tf.keras.optimizers.Adam(lr_schedule),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# ── Callbacks — no ReduceLROnPlateau (conflicts with custom schedule) ─────
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        f'{BASE_DIR}/best_model.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=15, restore_best_weights=True),
]

# ── Gradient check first ──────────────────────────────────────────────────
for xb, yb in train_ds.take(1):
    with tf.GradientTape() as tape:
        preds = model(xb, training=True)
        loss  = tf.reduce_mean(
                    tf.keras.losses.sparse_categorical_crossentropy(yb, preds))
    grads = tape.gradient(loss, model.trainable_variables)
    norms = [float(tf.norm(g)) for g in grads if g is not None]
    print(f"Loss: {float(loss):.4f}")
    print(f"Grad norms: {[f'{n:.4f}' for n in norms]}")
    print("GO" if not any(n == 0.0 for n in norms) else "STOP")

# ── Train ─────────────────────────────────────────────────────────────────
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    steps_per_epoch=steps_per_epoch,
    callbacks=callbacks
)

# ── Save ──────────────────────────────────────────────────────────────────
model.save(f'{BASE_DIR}/final_model.keras')
with open(f'{BASE_DIR}/history.json', 'w') as f:
    json.dump(history.history, f)

best_val = max(history.history['val_accuracy'])
best_ep  = history.history['val_accuracy'].index(best_val) + 1
print(f"\nBest val_accuracy: {best_val:.4f} at epoch {best_ep}")

Loss: 5.5377
Grad norms: ['0.3582', '0.0290', '0.3975', '0.1637', '0.0446', '0.4209', '0.1702', '0.0478', '0.3368', '0.1591', '0.0682', '0.3474', '0.1515', '0.0710', '0.2871', '0.0950', '0.2827', '0.1150']
GO
Epoch 1/100
1252/1254 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.0049 - loss: 5.5355
Epoch 1: val_accuracy improved from None to 0.02053, saving model to /content/drive/MyDrive/asl-signs/best_model.keras

Epoch 1: finished saving model to /content/drive/MyDrive/asl-signs/best_model.keras
1254/1254 ━━━━━━━━━━━━━━━━━━━━ 37s 25ms/step - accuracy: 0.0071 - loss: 5.4645 - val_accuracy: 0.0205 - val_loss: 5.1757
Epoch 2/100
1253/1254 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.0241 - loss: 5.0867
Epoch 2: val_accuracy improved from 0.02053 to 0.05370, saving model to /content/drive/MyDrive/asl-signs/best_model.keras

Epoch 2: finished saving model to /content/drive/MyDrive/asl-signs/best_model.keras
1254/1254 ━━━━━━━━━━━━━━━━━━━━ 31s 24ms/step - accuracy: 0.0326 - loss: 4.9444 

In [15]:
# ── Stronger augmentation ──────────────────────────
def augment(x, y):
    x = tf.cast(x, tf.float32)

    # 1. Gaussian noise
    x = x + tf.random.normal(tf.shape(x), stddev=0.02)

    # 2. Horizontal flip
    x = tf.cond(tf.random.uniform(()) > 0.5,
                lambda: x * tf.constant([[[-1., 1., 1.]]]),
                lambda: x)

    # 3. Scale jitter — random body size variation
    scale = tf.random.uniform((), 0.8, 1.2)
    x = x * scale

    # 4. Time masking — randomly zero out up to 4 consecutive frames
    start = tf.random.uniform((), 0, 28, dtype=tf.int32)
    length = tf.random.uniform((), 1, 5, dtype=tf.int32)
    mask = tf.ones((32, 75, 3))
    indices = tf.range(32)
    time_mask = tf.cast(
        (indices < start) | (indices >= start + length),
        tf.float32)
    x = x * time_mask[:, tf.newaxis, tf.newaxis]

    # 5. Landmark dropout — randomly zero out entire landmarks
    landmark_mask = tf.cast(
        tf.random.uniform((75,)) > 0.1,  # drop 10% of landmarks
        tf.float32)
    x = x * landmark_mask[tf.newaxis, :, tf.newaxis]

    return x, y

# Convert integer labels to one-hot encoding for CategoricalCrossentropy
y_train_one_hot = tf.one_hot(y_train, depth=NUM_CLASSES)
y_val_one_hot   = tf.one_hot(y_val,   depth=NUM_CLASSES)

# ── Rebuild datasets with new augmentation ───────────────────
train_ds = (tf.data.Dataset.from_tensor_slices((X_train, y_train_one_hot))
            .shuffle(10000)
            .batch(BATCH_SIZE)
            .map(augment, num_parallel_calls=tf.data.AUTOTUNE)
            .repeat()
            .prefetch(tf.data.AUTOTUNE))

val_ds = (tf.data.Dataset.from_tensor_slices((X_val, y_val_one_hot))
          .batch(BATCH_SIZE)
          .map(lambda x, y: (tf.cast(x, tf.float32), y),
               num_parallel_calls=tf.data.AUTOTUNE)
          .prefetch(tf.data.AUTOTUNE))

# ── Model with stronger regularization ────────────────────
def build_model(num_classes):
    inp = tf.keras.Input(shape=(32, 75, 3))
    x   = tf.keras.layers.Reshape((32, 75*3))(inp)
    x   = tf.keras.layers.Dense(256, activation='relu',
              kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x   = tf.keras.layers.Dropout(0.3)(x)
    x   = tf.keras.layers.Bidirectional(
              tf.keras.layers.LSTM(256, return_sequences=True))(x)  # no recurrent_dropout
    x   = tf.keras.layers.Bidirectional(
              tf.keras.layers.LSTM(128))(x)                         # no recurrent_dropout
    x   = tf.keras.layers.Dense(256, activation='relu',
              kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x   = tf.keras.layers.Dropout(0.4)(x)
    out = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
    return tf.keras.Model(inp, out)

model = build_model(NUM_CLASSES)

# ── Label smoothing — prevents overconfident predictions ─────
model.compile(
    optimizer=tf.keras.optimizers.Adam(lr_schedule),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

# ── Same callbacks ──────────────────────────
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        f'{BASE_DIR}/best_model_v2.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=15, restore_best_weights=True),
]

# ── Rebuild LR schedule for fresh model ───────────────
EPOCHS = 100
steps_per_epoch = len(X_train) // BATCH_SIZE
total_steps = EPOCHS * steps_per_epoch
warmup_steps = 5 * steps_per_epoch

lr_schedule = WarmupCosineDecay(
    peak_lr=1e-3,
    total_steps=total_steps,
    warmup_steps=warmup_steps
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(lr_schedule),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    steps_per_epoch=steps_per_epoch,
    callbacks=callbacks
)

# Save
model.save(f'{BASE_DIR}/final_model_v2.keras')
with open(f'{BASE_DIR}/history_v2.json', 'w') as f:
    json.dump(history.history, f)

best_val = max(history.history['val_accuracy'])
best_ep  = history.history['val_accuracy'].index(best_val) + 1
print(f"\nBest val_accuracy: {best_val:.4f} at epoch {best_ep}")

Epoch 1/100
1253/1254 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.0050 - loss: 5.5690
Epoch 1: val_accuracy improved from None to 0.01425, saving model to /content/drive/MyDrive/asl-signs/best_model_v2.keras

Epoch 1: finished saving model to /content/drive/MyDrive/asl-signs/best_model_v2.keras
1254/1254 ━━━━━━━━━━━━━━━━━━━━ 39s 28ms/step - accuracy: 0.0067 - loss: 5.5309 - val_accuracy: 0.0143 - val_loss: 5.3375
Epoch 2/100
1254/1254 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.0171 - loss: 5.3103
Epoch 2: val_accuracy improved from 0.01425 to 0.03154, saving model to /content/drive/MyDrive/asl-signs/best_model_v2.keras

Epoch 2: finished saving model to /content/drive/MyDrive/asl-signs/best_model_v2.keras
1254/1254 ━━━━━━━━━━━━━━━━━━━━ 33s 26ms/step - accuracy: 0.0213 - loss: 5.2390 - val_accuracy: 0.0315 - val_loss: 5.0828
Epoch 3/100
1252/1254 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.0367 - loss: 5.0406
Epoch 3: val_accuracy improved from 0.03154 to 0.07649, saving mo

KeyboardInterrupt: 

# Transformer

In [17]:
# ── Transformer model ─────────────────────────────────────────────────────
def build_transformer_model(num_classes, num_frames=32, num_landmarks=75,
                             d_model=256, num_heads=4, num_layers=2, dff=512):
    inp = tf.keras.Input(shape=(num_frames, num_landmarks, 3))

    # Flatten landmarks per frame → (batch, 32, 225)
    x = tf.keras.layers.Reshape((num_frames, num_landmarks*3))(inp)

    # Project to d_model
    x = tf.keras.layers.Dense(d_model)(x)

    # Positional encoding
    positions = tf.range(num_frames)
    pos_emb   = tf.keras.layers.Embedding(num_frames, d_model)(positions)
    x = x + pos_emb

    # Transformer encoder blocks
    for _ in range(num_layers):
        # Multi-head self attention
        attn_out = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=d_model//num_heads,
            dropout=0.1)(x, x)
        x = tf.keras.layers.LayerNormalization()(x + attn_out)

        # Feed forward
        ff = tf.keras.layers.Dense(dff, activation='relu')(x)
        ff = tf.keras.layers.Dropout(0.1)(ff)
        ff = tf.keras.layers.Dense(d_model)(ff)
        x  = tf.keras.layers.LayerNormalization()(x + ff)

    # Global average pooling over time
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    x = tf.keras.layers.Dense(256, activation='relu',
            kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    out = tf.keras.layers.Dense(num_classes, activation='softmax')(x)

    return tf.keras.Model(inp, out)

# ── Simpler augmentation — noise + flip only, no time masking ────────────
def augment(x, y):
    x = tf.cast(x, tf.float32)
    x = x + tf.random.normal(tf.shape(x), stddev=0.02)
    x = tf.cond(tf.random.uniform(()) > 0.5,
                lambda: x * tf.constant([[[-1., 1., 1.]]]),
                lambda: x)
    return x, y

# ── Rebuild datasets ──────────────────────────────────────────────────────
train_ds = (tf.data.Dataset.from_tensor_slices((X_train, y_train))
            .shuffle(10000)
            .batch(BATCH_SIZE)
            .map(augment, num_parallel_calls=tf.data.AUTOTUNE)
            .repeat()
            .prefetch(tf.data.AUTOTUNE))

val_ds = (tf.data.Dataset.from_tensor_slices((X_val, y_val))
          .batch(BATCH_SIZE)
          .map(lambda x, y: (tf.cast(x, tf.float32), y),
               num_parallel_calls=tf.data.AUTOTUNE)
          .prefetch(tf.data.AUTOTUNE))

# # ── Fresh LR schedule ─────────────────────────────────────────────────────
# EPOCHS = 100
# steps_per_epoch = len(X_train) // BATCH_SIZE
# total_steps     = EPOCHS * steps_per_epoch
# warmup_steps    = 5 * steps_per_epoch

# lr_schedule = WarmupCosineDecay(
#     peak_lr=1e-3,
#     total_steps=total_steps,
#     warmup_steps=warmup_steps
# )

# model = build_transformer_model(NUM_CLASSES)
# model.summary()

# callbacks = [
#     tf.keras.callbacks.ModelCheckpoint(
#         f'{BASE_DIR}/best_model_transformer.keras',
#         monitor='val_accuracy', save_best_only=True, verbose=1),
#     tf.keras.callbacks.EarlyStopping(
#         monitor='val_accuracy', patience=15, restore_best_weights=True),
# ]

# history = model.fit(
#     train_ds,
#     validation_data=val_ds,
#     epochs=EPOCHS,
#     steps_per_epoch=steps_per_epoch,
#     callbacks=callbacks
# )

# model.save(f'{BASE_DIR}/final_model_transformer.keras')
# with open(f'{BASE_DIR}/history_transformer.json', 'w') as f:
#     json.dump(history.history, f)

# best_val = max(history.history['val_accuracy'])
# best_ep  = history.history['val_accuracy'].index(best_val) + 1
# print(f"\nBest val_accuracy: {best_val:.4f} at epoch {best_ep}")

Model: "functional_9"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_9       │ (None, 32, 75, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_9 (Reshape) │ (None, 32, 225)   │          0 │ input_layer_9[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_30 (Dense)    │ (None, 32, 256)   │     57,856 │ reshape_9[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_5 (Add)         │ (None, 32, 256)   │          0 │ dense_30[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 32, 256)   │    263,168 │ add_5[0][0],      │
│ (MultiHeadAttentio… │                   │            │ add_5[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_6 (Add)         │ (None, 32, 256)   │          0 │ add_5[0][0],      │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 32, 256)   │        512 │ add_6[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_31 (Dense)    │ (None, 32, 512)   │    131,584 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_19          │ (None, 32, 512)   │          0 │ dense_31[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_32 (Dense)    │ (None, 32, 256)   │    131,328 │ dropout_19[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_7 (Add)         │ (None, 32, 256)   │          0 │ layer_normalizat… │
│                     │                   │            │ dense_32[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 32, 256)   │        512 │ add_7[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 32, 256)   │    263,168 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_8 (Add)         │ (None, 32, 256)   │          0 │ layer_normalizat… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 32, 256)   │        512 │ add_8[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_33 (Dense)    │ (None, 32, 512)   │    131,584 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_21          │ (None, 32, 512)   │          0 │ dense_33[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_34 (Dense)    │ (None, 32, 256)   │    131,328 │ dropout_21[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_9 (Add)         │ (None, 32, 256)   │          0 │ layer_normalizat… │
│                     │                   │            │ dense_34[0][0]  

 Total params: 1,242,106 (4.74 MB)

 Trainable params: 1,242,106 (4.74 MB)

 Non-trainable params: 0 (0.00 B)

ValueError: You must call `compile()` before using the model.

In [18]:
# ── Complete Transformer Training Cell ───────────────────────────────────
EPOCHS = 100
steps_per_epoch = len(X_train) // BATCH_SIZE
total_steps     = EPOCHS * steps_per_epoch
warmup_steps    = 5 * steps_per_epoch

lr_schedule = WarmupCosineDecay(
    peak_lr=1e-3,
    total_steps=total_steps,
    warmup_steps=warmup_steps
)

model = build_transformer_model(NUM_CLASSES)
model.compile(
    optimizer=tf.keras.optimizers.Adam(lr_schedule),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        f'{BASE_DIR}/best_model_transformer.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=15, restore_best_weights=True),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    steps_per_epoch=steps_per_epoch,
    callbacks=callbacks
)

model.save(f'{BASE_DIR}/final_model_transformer.keras')
with open(f'{BASE_DIR}/history_transformer.json', 'w') as f:
    json.dump(history.history, f)

best_val = max(history.history['val_accuracy'])
best_ep  = history.history['val_accuracy'].index(best_val) + 1
print(f"\nBest val_accuracy: {best_val:.4f} at epoch {best_ep}")

Epoch 1/100
1251/1254 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.0040 - loss: 5.6359
Epoch 1: val_accuracy improved from None to 0.01200, saving model to /content/drive/MyDrive/asl-signs/best_model_transformer.keras

Epoch 1: finished saving model to /content/drive/MyDrive/asl-signs/best_model_transformer.keras
1254/1254 ━━━━━━━━━━━━━━━━━━━━ 38s 18ms/step - accuracy: 0.0056 - loss: 5.5428 - val_accuracy: 0.0120 - val_loss: 5.2663
Epoch 2/100
1252/1254 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.0122 - loss: 5.2388
Epoch 2: val_accuracy improved from 0.01200 to 0.02434, saving model to /content/drive/MyDrive/asl-signs/best_model_transformer.keras

Epoch 2: finished saving model to /content/drive/MyDrive/asl-signs/best_model_transformer.keras
1254/1254 ━━━━━━━━━━━━━━━━━━━━ 28s 16ms/step - accuracy: 0.0152 - loss: 5.1623 - val_accuracy: 0.0243 - val_loss: 4.9459
Epoch 3/100
1251/1254 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.0334 - loss: 4.8333
Epoch 3: val_accuracy improve

In [22]:
import math

class WarmupCosineDecay(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, peak_lr, total_steps, warmup_steps):
        self.peak_lr      = peak_lr
        self.total_steps  = total_steps
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        step      = tf.cast(step, tf.float32)
        warmup_lr = self.peak_lr * (step / self.warmup_steps)
        cosine_lr = self.peak_lr * 0.5 * (
            1 + tf.cos(math.pi * (step - self.warmup_steps)
                       / (self.total_steps - self.warmup_steps)))
        return tf.where(step < self.warmup_steps, warmup_lr, cosine_lr)

    def get_config(self):
        return {'peak_lr': self.peak_lr,
                'total_steps': self.total_steps,
                'warmup_steps': self.warmup_steps}

# Load with explicit custom_objects
model = tf.keras.models.load_model(
    f'{BASE_DIR}/best_model_transformer.keras',
    custom_objects={'WarmupCosineDecay': WarmupCosineDecay}
)
print("Model loaded OK")

# Fine-tuning schedule
EPOCHS_2        = 100
steps_per_epoch = len(X_train) // BATCH_SIZE
total_steps     = EPOCHS_2 * steps_per_epoch
warmup_steps    = 2 * steps_per_epoch

lr_schedule_2 = WarmupCosineDecay(
    peak_lr=3e-4,
    total_steps=total_steps,
    warmup_steps=warmup_steps
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(lr_schedule_2),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        f'{BASE_DIR}/best_model_transformer_v2.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=20, restore_best_weights=True),
]

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_2,
    steps_per_epoch=steps_per_epoch,
    callbacks=callbacks
)

with open(f'{BASE_DIR}/history_transformer_v2.json', 'w') as f:
    json.dump(history2.history, f)

best_val = max(history2.history['val_accuracy'])
best_ep  = history2.history['val_accuracy'].index(best_val) + 1
print(f"\nBest val_accuracy: {best_val:.4f} at epoch {best_ep}")

Model loaded OK
Epoch 1/100
1254/1254 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8436 - loss: 0.5617
Epoch 1: val_accuracy improved from None to 0.71648, saving model to /content/drive/MyDrive/asl-signs/best_model_transformer_v2.keras

Epoch 1: finished saving model to /content/drive/MyDrive/asl-signs/best_model_transformer_v2.keras
1254/1254 ━━━━━━━━━━━━━━━━━━━━ 37s 19ms/step - accuracy: 0.8375 - loss: 0.5819 - val_accuracy: 0.7165 - val_loss: 1.4582
Epoch 2/100
1252/1254 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.8201 - loss: 0.6408
Epoch 2: val_accuracy did not improve from 0.71648
1254/1254 ━━━━━━━━━━━━━━━━━━━━ 28s 17ms/step - accuracy: 0.8128 - loss: 0.6645 - val_accuracy: 0.6986 - val_loss: 1.4936
Epoch 3/100
1253/1254 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.7984 - loss: 0.7108
Epoch 3: val_accuracy did not improve from 0.71648
1254/1254 ━━━━━━━━━━━━━━━━━━━━ 20s 16ms/step - accuracy: 0.7976 - loss: 0.7154 - val_accuracy: 0.7096 - val_loss: 1.4181
Epoch 4/100
125

In [23]:
# Check if PyTorch Geometric is available or if we should use pure TF/Keras GCN
import subprocess
result = subprocess.run(['pip', 'show', 'torch'], capture_output=True, text=True)
print(result.stdout if result.stdout else "PyTorch not installed")

import tensorflow as tf
print(f"TF version: {tf.__version__}")
import sys
print(f"Python: {sys.version}")

Name: torch
Version: 2.10.0+cu128
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org
Author: 
Author-email: PyTorch Team <packages@pytorch.org>
License: BSD-3-Clause
Location: /usr/local/lib/python3.12/dist-packages
Requires: cuda-bindings, filelock, fsspec, jinja2, networkx, nvidia-cublas-cu12, nvidia-cuda-cupti-cu12, nvidia-cuda-nvrtc-cu12, nvidia-cuda-runtime-cu12, nvidia-cudnn-cu12, nvidia-cufft-cu12, nvidia-cufile-cu12, nvidia-curand-cu12, nvidia-cusolver-cu12, nvidia-cusparse-cu12, nvidia-cusparselt-cu12, nvidia-nccl-cu12, nvidia-nvjitlink-cu12, nvidia-nvshmem-cu12, nvidia-nvtx-cu12, setuptools, sympy, triton, typing-extensions
Required-by: accelerate, fastai, peft, sentence-transformers, timm, torchaudio, torchdata, torchvision

TF version: 2.19.0
Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]


# GCN

In [24]:
# Install PyTorch Geometric
import subprocess

subprocess.run(['pip', 'install', '-q', 'torch-geometric'], check=True)
subprocess.run(['pip', 'install', '-q', 'torch-scatter', 'torch-sparse',
                '-f', 'https://data.pyg.org/whl/torch-2.10.0+cu128.html'],
               check=True)

import torch
import torch_geometric
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0)}")
print(f"PyG: {torch_geometric.__version__}")

PyTorch: 2.10.0+cu128
CUDA available: True
Device: Tesla T4
PyG: 2.7.0


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.data import Data, DataLoader
import numpy as np
import json
import os
from sklearn.model_selection import train_test_split

# ── Constants ─────────────────────────────────────────────────────────────
NUM_FRAMES   = 32
NUM_LANDMARKS = 75
NUM_CLASSES  = 250
BATCH_SIZE   = 64
EPOCHS       = 100
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BASE_DIR     = '/content/drive/MyDrive/asl-signs'
print(f"Device: {DEVICE}")

# ── Skeleton edges ─────────────────────────────────────────────────────────
# Left hand (0-20), Pose (21-53), Right hand (54-74)
def get_edges():
    edges = []

    # Left hand finger connections
    hand_connections = [
        (0,1),(1,2),(2,3),(3,4),       # thumb
        (0,5),(5,6),(6,7),(7,8),       # index
        (0,9),(9,10),(10,11),(11,12),  # middle
        (0,13),(13,14),(14,15),(15,16),# ring
        (0,17),(17,18),(18,19),(19,20),# pinky
        (5,9),(9,13),(13,17)           # palm
    ]
    # Left hand (offset 0)
    for a, b in hand_connections:
        edges += [(a, b), (b, a)]

    # Right hand (offset 54)
    for a, b in hand_connections:
        edges += [(a+54, b+54), (b+54, a+54)]

    # Pose connections (offset 21)
    pose_connections = [
        (0,1),(1,2),(2,3),(3,4),       # nose to eyes/ears
        (0,5),(5,6),(6,7),(7,8),
        (9,10),                         # mouth
        (11,12),                        # shoulders
        (11,13),(13,15),               # left arm
        (12,14),(14,16),               # right arm
        (11,23),(12,24),(23,24),       # torso
        (23,25),(25,27),(24,26),(26,28),# legs
    ]
    for a, b in pose_connections:
        if a < 33 and b < 33:
            edges += [(a+21, b+21), (b+21, a+21)]

    # Cross connections — wrists to hands
    # Left wrist (pose landmark 15 = index 21+15=36) to left hand root (0)
    edges += [(36, 0), (0, 36)]
    # Right wrist (pose landmark 16 = index 21+16=37) to right hand root (54)
    edges += [(37, 54), (54, 37)]

    return edges

edges     = get_edges()
edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous().to(DEVICE)
print(f"Edge index shape: {edge_index.shape}")
print(f"Total edges: {edge_index.shape[1]}")

# ── GCN Model ─────────────────────────────────────────────────────────────
class SignGCN(nn.Module):
    def __init__(self, in_channels=3, hidden=128, num_classes=NUM_CLASSES,
                 num_frames=NUM_FRAMES):
        super().__init__()
        self.num_frames   = num_frames
        self.num_landmarks = NUM_LANDMARKS

        # Spatial GCN layers — process each frame's graph
        self.gcn1 = GCNConv(in_channels, hidden)
        self.gcn2 = GCNConv(hidden, hidden)
        self.gcn3 = GCNConv(hidden, hidden)

        # Temporal Transformer — process sequence of frame embeddings
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden, nhead=4, dim_feedforward=256,
            dropout=0.1, batch_first=True)
        self.temporal = nn.TransformerEncoder(encoder_layer, num_layers=2)

        # Positional encoding for temporal
        self.pos_emb = nn.Embedding(num_frames, hidden)

        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(hidden, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x, edge_index):
        # x: (batch, frames, landmarks, 3)
        B, T, N, C = x.shape

        # Process each frame with GCN
        # Reshape to (batch*frames, landmarks, 3)
        x = x.view(B * T, N, C)

        # Process graph per sample-frame
        frame_embeddings = []
        for i in range(B * T):
            node_feat = x[i]  # (75, 3)
            h = F.relu(self.gcn1(node_feat, edge_index))
            h = F.dropout(h, p=0.1, training=self.training)
            h = F.relu(self.gcn2(h, edge_index))
            h = F.dropout(h, p=0.1, training=self.training)
            h = self.gcn3(h, edge_index)
            # Global mean pool over landmarks
            h = h.mean(dim=0)  # (hidden,)
            frame_embeddings.append(h)

        # Stack to (batch*frames, hidden) then reshape to (batch, frames, hidden)
        x = torch.stack(frame_embeddings, dim=0).view(B, T, -1)

        # Add positional encoding
        positions = torch.arange(T, device=x.device)
        x = x + self.pos_emb(positions)

        # Temporal transformer
        x = self.temporal(x)  # (batch, frames, hidden)

        # Global average pool over time
        x = x.mean(dim=1)  # (batch, hidden)

        return self.classifier(x)

# ── Dataset ───────────────────────────────────────────────────────────────
class ASLDataset(torch.utils.data.Dataset):
    def __init__(self, X, y, augment=False):
        self.X       = torch.tensor(X, dtype=torch.float32)
        self.y       = torch.tensor(y, dtype=torch.long)
        self.augment = augment

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = self.X[idx]  # (32, 75, 3)
        y = self.y[idx]
        if self.augment:
            # Noise
            x = x + torch.randn_like(x) * 0.02
            # Horizontal flip
            if torch.rand(1) > 0.5:
                x = x * torch.tensor([[[-1., 1., 1.]]])
        return x, y

# ── Load data ─────────────────────────────────────────────────────────────
print("Loading data...")
X = np.load(f'{BASE_DIR}/X.npy', mmap_mode='r')
y = np.load(f'{BASE_DIR}/y.npy')

idx = np.arange(len(X))
idx_train, idx_val = train_test_split(
    idx, test_size=0.15, random_state=42, stratify=y)

X_train = X[idx_train].astype(np.float32)
X_val   = X[idx_val].astype(np.float32)
y_train = y[idx_train].astype(np.int64)
y_val   = y[idx_val].astype(np.int64)

del X, idx, idx_train, idx_val
import gc; gc.collect()

print(f"Train: {X_train.shape}  Val: {X_val.shape}")

train_dataset = ASLDataset(X_train, y_train, augment=True)
val_dataset   = ASLDataset(X_val,   y_val,   augment=False)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True)
val_loader = torch.utils.data.DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True)

# ── Training ──────────────────────────────────────────────────────────────
model     = SignGCN().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=1e-6)
criterion = nn.CrossEntropyLoss()

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

best_val_acc = 0.0
history      = {'train_acc': [], 'val_acc': [], 'train_loss': [], 'val_loss': []}

for epoch in range(1, EPOCHS + 1):
    # Train
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        out  = model(xb, edge_index)
        loss = criterion(out, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss    += loss.item() * len(yb)
        train_correct += (out.argmax(1) == yb).sum().item()
        train_total   += len(yb)

    # Validate
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            out  = model(xb, edge_index)
            loss = criterion(out, yb)
            val_loss    += loss.item() * len(yb)
            val_correct += (out.argmax(1) == yb).sum().item()
            val_total   += len(yb)

    train_acc = train_correct / train_total
    val_acc   = val_correct   / val_total
    t_loss    = train_loss    / train_total
    v_loss    = val_loss      / val_total

    scheduler.step()

    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['train_loss'].append(t_loss)
    history['val_loss'].append(v_loss)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(),
                   f'{BASE_DIR}/best_gcn_model.pth')
        print(f"Epoch {epoch:3d}: train={train_acc:.4f} val={val_acc:.4f} "
              f"loss={v_loss:.4f} ← saved")
    else:
        print(f"Epoch {epoch:3d}: train={train_acc:.4f} val={val_acc:.4f} "
              f"loss={v_loss:.4f}")

    # Early stopping
    if epoch > 20 and all(
        history['val_acc'][-i] <= best_val_acc - 0.005
        for i in range(1, 16)
    ):
        print(f"Early stopping at epoch {epoch}")
        break

with open(f'{BASE_DIR}/history_gcn.json', 'w') as f:
    json.dump(history, f)

print(f"\nBest val_accuracy: {best_val_acc:.4f}")

Device: cuda
Edge index shape: torch.Size([2, 138])
Total edges: 138
Loading data...


# Ensemble BILSTM + Transformer

In [3]:
import numpy as np
import tensorflow as tf
import json
from sklearn.model_selection import train_test_split

# ── Load both models ───────────────────────────────────────────────────────
import math

class WarmupCosineDecay(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, peak_lr, total_steps, warmup_steps):
        self.peak_lr      = peak_lr
        self.total_steps  = total_steps
        self.warmup_steps = warmup_steps
    def __call__(self, step):
        step      = tf.cast(step, tf.float32)
        warmup_lr = self.peak_lr * (step / self.warmup_steps)
        cosine_lr = self.peak_lr * 0.5 * (
            1 + tf.cos(math.pi * (step - self.warmup_steps)
                       / (self.total_steps - self.warmup_steps)))
        return tf.where(step < self.warmup_steps, warmup_lr, cosine_lr)
    def get_config(self):
        return {'peak_lr': self.peak_lr,
                'total_steps': self.total_steps,
                'warmup_steps': self.warmup_steps}

print("Loading models...")
transformer = tf.keras.models.load_model(
    f'{BASE_DIR}/best_model_transformer.keras',
    custom_objects={'WarmupCosineDecay': WarmupCosineDecay})
print(f"Transformer loaded ✅")

bilstm = tf.keras.models.load_model(
    f'{BASE_DIR}/best_model.keras',
    custom_objects={'WarmupCosineDecay': WarmupCosineDecay})
print(f"BiLSTM loaded ✅")

# ── Rebuild val set ────────────────────────────────────────────────────────
X = np.load(f'{BASE_DIR}/X.npy', mmap_mode='r')
y = np.load(f'{BASE_DIR}/y.npy')

idx = np.arange(len(X))
idx_train, idx_val = train_test_split(
    idx, test_size=0.15, random_state=42, stratify=y)

X_val = X[idx_val].astype(np.float32)
y_val = y[idx_val].astype(np.int32)

del X, idx, idx_train
import gc; gc.collect()
print(f"Val set: {X_val.shape}")

# ── Get predictions from both models ──────────────────────────────────────
print("\nGetting Transformer predictions...")
transformer_probs = transformer.predict(X_val, batch_size=128, verbose=1)

print("\nGetting BiLSTM predictions...")
bilstm_probs = bilstm.predict(X_val, batch_size=128, verbose=1)

# ── Individual accuracies ─────────────────────────────────────────────────
transformer_preds = np.argmax(transformer_probs, axis=1)
bilstm_preds      = np.argmax(bilstm_probs,      axis=1)

transformer_acc = (transformer_preds == y_val).mean()
bilstm_acc      = (bilstm_preds      == y_val).mean()
print(f"\nTransformer accuracy: {transformer_acc:.4f}")
print(f"BiLSTM accuracy:      {bilstm_acc:.4f}")

# ── Grid search best ensemble weights ─────────────────────────────────────
print("\nSearching best ensemble weights...")
best_acc    = 0.0
best_weight = 0.0

for w in np.arange(0.0, 1.05, 0.05):
    ensemble_probs = w * transformer_probs + (1 - w) * bilstm_probs
    ensemble_preds = np.argmax(ensemble_probs, axis=1)
    acc = (ensemble_preds == y_val).mean()
    if acc > best_acc:
        best_acc    = acc
        best_weight = w
    print(f"  w={w:.2f} transformer + {1-w:.2f} bilstm → {acc:.4f}")

print(f"\nBest ensemble: {best_weight:.2f} × Transformer + "
      f"{1-best_weight:.2f} × BiLSTM → {best_acc:.4f}")

# ── Top-3 accuracy of best ensemble ───────────────────────────────────────
best_probs   = best_weight * transformer_probs + (1-best_weight) * bilstm_probs
top3_correct = sum(
    y_val[i] in np.argsort(best_probs[i])[-3:]
    for i in range(len(y_val))
)
top3_acc = top3_correct / len(y_val)
print(f"Top-3 accuracy: {top3_acc:.4f}")

# ── Save ensemble weights ─────────────────────────────────────────────────
ensemble_config = {
    'transformer_weight': float(best_weight),
    'bilstm_weight':      float(1 - best_weight),
    'transformer_acc':    float(transformer_acc),
    'bilstm_acc':         float(bilstm_acc),
    'ensemble_acc':       float(best_acc),
    'top3_acc':           float(top3_acc)
}
with open(f'{BASE_DIR}/ensemble_config.json', 'w') as f:
    json.dump(ensemble_config, f, indent=2)

print(f"\nEnsemble config saved to Drive")
print(f"\n{'='*45}")
print(f"Summary:")
print(f"  BiLSTM:      {bilstm_acc:.4f}")
print(f"  Transformer: {transformer_acc:.4f}")
print(f"  Ensemble:    {best_acc:.4f}  ← best")
print(f"  Top-3:       {top3_acc:.4f}")
print(f"{'='*45}")

Loading models...
Transformer loaded ✅
BiLSTM loaded ✅
Val set: (14172, 32, 75, 3)

Getting Transformer predictions...
111/111 ━━━━━━━━━━━━━━━━━━━━ 7s 33ms/step

Getting BiLSTM predictions...
111/111 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step

Transformer accuracy: 0.7269
BiLSTM accuracy:      0.6483

Searching best ensemble weights...
  w=0.00 transformer + 1.00 bilstm → 0.6483
  w=0.05 transformer + 0.95 bilstm → 0.6522
  w=0.10 transformer + 0.90 bilstm → 0.6564
  w=0.15 transformer + 0.85 bilstm → 0.6614
  w=0.20 transformer + 0.80 bilstm → 0.6674
  w=0.25 transformer + 0.75 bilstm → 0.6762
  w=0.30 transformer + 0.70 bilstm → 0.6853
  w=0.35 transformer + 0.65 bilstm → 0.6957
  w=0.40 transformer + 0.60 bilstm → 0.7084
  w=0.45 transformer + 0.55 bilstm → 0.7202
  w=0.50 transformer + 0.50 bilstm → 0.7369
  w=0.55 transformer + 0.45 bilstm → 0.7406
  w=0.60 transformer + 0.40 bilstm → 0.7422
  w=0.65 transformer + 0.35 bilstm → 0.7420
  w=0.70 transformer + 0.30 bilstm → 0.7391
  w=0.75 tr

# TFLite

In [6]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
import tensorflow as tf
import numpy as np
from google.colab import drive

drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/asl-signs'

# Register custom class
@tf.keras.utils.register_keras_serializable()
class WarmupCosineDecay(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, peak_lr, total_steps, warmup_steps, **kwargs):
        super().__init__(**kwargs)
        self.peak_lr = peak_lr
        self.total_steps = total_steps
        self.warmup_steps = warmup_steps
    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warmup = self.peak_lr * (step / self.warmup_steps)
        cosine = self.peak_lr * 0.5 * (1 + tf.cos(3.14159 * (step - self.warmup_steps) / (self.total_steps - self.warmup_steps)))
        return tf.where(step < self.warmup_steps, warmup, cosine)
    def get_config(self):
        return {'peak_lr': self.peak_lr, 'total_steps': self.total_steps, 'warmup_steps': self.warmup_steps}

model = tf.keras.models.load_model(
    f'{BASE_DIR}/final_model_transformer.keras',
    custom_objects={'WarmupCosineDecay': WarmupCosineDecay}
)
print("Model loaded:", model.input_shape)

# Convert to float32 TFLite — NO quantization, NO int8
@tf.function(input_signature=[tf.TensorSpec(shape=[1, 32, 75, 3], dtype=tf.float32)])
def serving_fn(x):
    return model(x, training=False)

converter = tf.lite.TFLiteConverter.from_concrete_functions(
    [serving_fn.get_concrete_function()], model
)
# NO optimizations — pure float32
tflite_model = converter.convert()

out_path = f'{BASE_DIR}/transformer_float32.tflite'
with open(out_path, 'wb') as f:
    f.write(tflite_model)
print(f"Saved: {out_path}  ({len(tflite_model)/1024:.1f} KB)")

# Verify
interp = tf.lite.In

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model loaded: (None, 32, 75, 3)
Saved: /content/drive/MyDrive/asl-signs/transformer_float32.tflite  (4903.8 KB)


AttributeError: module 'tensorflow._api.v2.lite' has no attribute 'In'

In [9]:
import numpy as np, json, tensorflow as tf

BASE_DIR = '/content/drive/MyDrive/asl-signs'

X = np.load(f'{BASE_DIR}/X.npy', mmap_mode='r')
y = np.load(f'{BASE_DIR}/y.npy')
X_test = X[:100].astype(np.float32)
y_test = y[:100]

interp = tf.lite.Interpreter(model_path=f'{BASE_DIR}/transformer_float32.tflite')
interp.allocate_tensors()
inp = interp.get_input_details()[0]['index']
out = interp.get_output_details()[0]['index']

correct = 0
for i in range(100):
    interp.set_tensor(inp, X_test[i:i+1])
    interp.invoke()
    result = interp.get_tensor(out)[0]
    pred = np.argmax(result)
    conf = result[pred]
    correct += int(pred == y_test[i])

print(f'Accuracy: {correct}/100 = {correct}%')
print(f'Sample confidences: {[round(float(interp.get_tensor(out)[0].max()),3) for _ in range(1)]}')

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Accuracy: 90/100 = 90%
Sample confidences: [0.718]
